# 01 — Baseline Momentum

**Inertia, Momentum Timing** — Sprint 1

Goal: characterize the **static** momentum factor (UMD) before we try to time it.

1. Load Ken French UMD (1927–present) and FF5+UMD panel (1963–present).
2. Compute full-sample performance: cumulative return, Sharpe, max drawdown, skew, kurtosis.
3. Identify the famous crash episodes (1932, 2002, **March–May 2009**, 2022).
4. Plot cumulative return, drawdown, and rolling 3-year Sharpe — Inertia style.

Anchor: Jegadeesh-Titman (1993); Daniel-Moskowitz (2015).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import get_ff_momentum, get_factor_panel
from src.inertia_style import apply_style, C, FULL_COL, SINGLE_COL, save_fig

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 1. Load data

In [2]:
# Long-sample UMD (back to 1927)
umd_long = get_ff_momentum()

# FF5 + UMD panel (1963+) for later factor regressions
panel = get_factor_panel()

print(f'UMD (long):      {umd_long.index.min().date()} -> {umd_long.index.max().date()}  ({len(umd_long)} months)')
print(f'FF5+UMD panel:   {panel.index.min().date()} -> {panel.index.max().date()}  ({len(panel)} months)')
umd_long.tail()

UMD (long):      1927-01-31 -> 2026-02-28  (1190 months)
FF5+UMD panel:   1963-07-31 -> 2026-02-28  (752 months)


,UMD
date,
2025-10-31,0.0027
2025-11-30,-0.0180
2025-12-31,-0.0241
2026-01-31,0.0496
2026-02-28,0.0133


## 2. Performance summary

We report three samples:

- **Full** (1927–present): UMD only
- **Post-1963** (to align with FF5 for later regressions)
- **Post-2000** (modern era, where momentum crashes were most severe)

In [3]:
def perf_table(r: pd.Series, rf: pd.Series | None = None) -> dict:
    """Annualized performance stats for a monthly return series (decimal)."""
    r = r.dropna()
    if rf is not None:
        rf = rf.reindex(r.index).fillna(0)
    mean_a = r.mean() * 12
    vol_a  = r.std(ddof=1) * np.sqrt(12)
    sharpe = mean_a / vol_a if vol_a > 0 else np.nan
    # max drawdown on cumulative log return
    cum = (1 + r).cumprod()
    dd  = cum / cum.cummax() - 1
    mdd = dd.min()
    return {
        'n_months':      len(r),
        'start':         r.index.min().strftime('%Y-%m'),
        'end':           r.index.max().strftime('%Y-%m'),
        'mean_ann':      mean_a,
        'vol_ann':       vol_a,
        'sharpe_ann':    sharpe,
        'skew':          r.skew(),
        'excess_kurt':   r.kurt(),
        'max_drawdown':  mdd,
    }

samples = {
    'Full (1927+)':    umd_long['UMD'],
    'Post-1963':       umd_long['UMD'].loc['1963-07':],
    'Post-2000':       umd_long['UMD'].loc['2000-01':],
}
rows = {name: perf_table(r) for name, r in samples.items()}
perf_df = pd.DataFrame(rows).T
perf_df

,n_months,start,end,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Full (1927+),1190,1927-01,2026-02,0.0734,0.1621,0.4529,-3.0619,28.0632,-0.7862
Post-1963,752,1963-07,2026-02,0.0716,0.1445,0.4954,-1.3069,9.8365,-0.5782
Post-2000,314,2000-01,2026-02,0.0222,0.1748,0.1272,-1.4771,9.4840,-0.5782


### Observations

- Full-sample UMD Sharpe is respectable for a long-only equity factor, but...
- **Left-tail is brutal.** Skew is strongly negative and excess kurtosis is large — the distribution of UMD returns is not Gaussian.
- Max drawdowns are multi-year events: enough to destroy a compounding career.

This is exactly the distribution Daniel-Moskowitz (2015) attack.

## 3. Cumulative return chart

In [4]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.0))
r = umd_long['UMD']
cum = (1 + r).cumprod()

ax.plot(cum.index, cum.values, color=C['blue'], linewidth=1.1, label='Static UMD (long-short)')
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \$1 (log scale)')
ax.set_xlabel('')
ax.set_title('Static momentum: cumulative return, 1927–present', loc='left', color=C['dark'])
ax.legend(frameon=False, loc='upper left')

save_fig(fig, '01_umd_cumret_log', out_dir='../figures')
plt.show()

<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
/var/folders/z3/9yfkxffn0736bc71t6rrt9k00000gn/T/ipykernel_98777/3391020728.py:7: SyntaxWarning: invalid escape sequence '\$'
  ax.set_ylabel('Cumulative growth of \$1 (log scale)')


/Users/willwu/Documents/inertia-momentum-timing/src/inertia_style.py:95: UserWarning: Glyph 8211 (\N{EN DASH}) missing from font(s) cmr10.
  fig.savefig(f"{out_dir}/{name}.pdf")
/Users/willwu/Documents/inertia-momentum-timing/src/inertia_style.py:96: UserWarning: Glyph 8211 (\N{EN DASH}) missing from font(s) cmr10.
  fig.savefig(f"{out_dir}/{name}.png")


  saved: ../figures/01_umd_cumret_log.{pdf,png}


## 4. Drawdown chart — where the crashes live

In [5]:
fig, ax = plt.subplots(figsize=(FULL_COL, 2.6))
cum = (1 + umd_long['UMD']).cumprod()
dd  = cum / cum.cummax() - 1

ax.fill_between(dd.index, dd.values, 0, color=C['red'], alpha=0.35, linewidth=0)
ax.plot(dd.index, dd.values, color=C['red'], linewidth=0.8)
ax.axhline(0, color=C['muted'], linewidth=0.4)
ax.set_ylabel('Drawdown')
ax.set_title('Static momentum: drawdown, 1927–present', loc='left', color=C['dark'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

save_fig(fig, '02_umd_drawdown', out_dir='../figures')
plt.show()

/Users/willwu/Documents/inertia-momentum-timing/src/inertia_style.py:95: UserWarning: Glyph 8211 (\N{EN DASH}) missing from font(s) cmr10.
  fig.savefig(f"{out_dir}/{name}.pdf")
/Users/willwu/Documents/inertia-momentum-timing/src/inertia_style.py:96: UserWarning: Glyph 8211 (\N{EN DASH}) missing from font(s) cmr10.
  fig.savefig(f"{out_dir}/{name}.png")


  saved: ../figures/02_umd_drawdown.{pdf,png}


## 5. Worst months — the crashes we need to avoid

In [6]:
worst = umd_long['UMD'].nsmallest(15).to_frame('UMD_return')
worst.index = worst.index.strftime('%Y-%m')
worst['UMD_return_pct'] = worst['UMD_return'].map(lambda x: f'{x:.1%}')
worst[['UMD_return_pct']]

,UMD_return_pct
date,
1932-08,-52.6%
1932-07,-45.7%
2009-04,-34.3%
1939-09,-31.4%
2001-01,-25.4%
1938-06,-24.9%
1931-06,-17.6%
1933-04,-17.4%
2002-11,-16.4%


Pattern: the worst months cluster in (1) the 1932–1933 recovery from the Great Depression, (2) **March–April 2009** (recovery from the GFC), and (3) the post-COVID reopening.

Daniel-Moskowitz's observation: **momentum crashes when the market rebounds sharply from a prolonged drawdown**. The short-leg (recent losers) becomes heavily weighted toward high-beta, crisis-hit names — which explode upward in the recovery.

## 6. Rolling 3-year Sharpe

In [7]:
win = 36
rmean = umd_long['UMD'].rolling(win).mean() * 12
rvol  = umd_long['UMD'].rolling(win).std(ddof=1) * np.sqrt(12)
rsr   = rmean / rvol

fig, ax = plt.subplots(figsize=(FULL_COL, 2.6))
ax.plot(rsr.index, rsr.values, color=C['purple'], linewidth=1.0)
ax.axhline(0, color=C['muted'], linewidth=0.4)
ax.axhline(rsr.mean(), color=C['blue'], linewidth=0.6, linestyle='--',
           label=f'Mean = {rsr.mean():.2f}')
ax.set_ylabel('Rolling 3-year Sharpe')
ax.set_title('Static momentum: 3-year rolling Sharpe', loc='left', color=C['dark'])
ax.legend(frameon=False, loc='lower left')

save_fig(fig, '03_umd_rolling_sharpe', out_dir='../figures')
plt.show()

  saved: ../figures/03_umd_rolling_sharpe.{pdf,png}


## Takeaways for Sprint 1

- UMD has delivered a meaningful long-run Sharpe, but the distribution of monthly returns is decidedly non-Gaussian: strong negative skew and very fat tails.
- Crashes cluster in market-recovery episodes (1932, 2009, 2020–22).
- Rolling 3-year Sharpe is visibly cyclical — there are multi-year regimes where static momentum is a disaster.

This motivates the **dynamic momentum** strategy (Daniel-Moskowitz 2015), which is our academic benchmark in notebook 02.